# Orbit Wars — Training & Watching Self-Play

Bu notebook üç şeyi gösterir:

1. **PPO** ile küçük bir self-play eğitim koşusu
2. **DQN / Dueling DQN** alternatifleri
3. Eğitilmiş bir checkpoint ile **oyunu izleme** (matplotlib render)

## Modüller
- `rl.train` → PPO main loop
- `rl.dqn_train` → DQN / Dueling DQN main loop
- `rl.watch.play_self_play_game` → checkpoint yükle, kendine karşı oynat, render
- `rl.watch.load_any_checkpoint` → PPO veya DQN checkpoint otomatik tanı

## Komut satırı muadilleri (terminalde de çalıştırabilirsin)
```bash
python -m rl.train --iterations 50 --episodes-per-iter 4
python -m rl.dqn_train --iterations 50 --episodes-per-iter 4
python -m rl.dqn_train --dueling --iterations 50
python -m rl.watch --ckpt checkpoints/latest.pt --render-every 25
```

## 1. Kurulum

In [ ]:
from pathlib import Path
import sys

REPO = Path.cwd()
while REPO.name != 'Orbit_Wars' and REPO.parent != REPO:
    REPO = REPO.parent
sys.path.insert(0, str(REPO))
print('Repo:', REPO)

In [ ]:
import torch
import matplotlib.pyplot as plt

from rl.train import TrainConfig, train
from rl.ppo import PPOConfig
from rl.rewards import RewardConfig
from rl.dqn_train import DQNTrainConfig, train_dqn
from rl.dqn import DQNConfig
from rl.watch import play_self_play_game, load_any_checkpoint

print('Cihaz:', 'cuda' if torch.cuda.is_available() else 'cpu')

## 2. PPO Eğitimi (Küçük Konfig)

Notebook'ta hızlı koşmak için tiny config: `d_model=32`, `n_layers=2`, kısa episode, az iter.
Gerçek eğitim için CLI ile büyük config kullan:
```bash
python -m rl.train --iterations 500 --episodes-per-iter 8 --d-model 128 --n-layers 4
```

In [ ]:
ppo_train_cfg = TrainConfig(
    iterations=3,
    episodes_per_iter=1,
    max_episode_steps=80,
    d_model=32, n_heads=2, n_layers=2,
    checkpoint_every=3,
    checkpoint_dir='nb_ckpt_ppo',
    log_path='nb_ckpt_ppo/train_log.jsonl',
    seed=0,
)
ppo_cfg = PPOConfig(epochs=2, batch_size=32, entropy_coef=0.01)

policy = train(ppo_train_cfg, ppo_cfg, RewardConfig())
print('\nPPO eğitim bitti. Checkpoint:', 'nb_ckpt_ppo/latest.pt')

## 3. DQN Eğitimi

Off-policy: replay buffer + target network + ε-greedy. Aynı transformer encoder, **branching** Q heads (`q_src`, `q_tgt`, `q_ship`).

`--dueling` flag'i `Dueling DQN` mimarisine geçer (V + A decomposition).

In [ ]:
dqn_train_cfg = DQNTrainConfig(
    iterations=3,
    episodes_per_iter=1,
    max_episode_steps=80,
    d_model=32, n_heads=2, n_layers=2,
    batch_size=32,
    learn_starts=32,
    replay_capacity=2000,
    target_sync_interval=50,
    eps_decay_steps=500,
    checkpoint_every=3,
    checkpoint_dir='nb_ckpt_dqn',
    log_path='nb_ckpt_dqn/train_log.jsonl',
    seed=0,
    dueling=False,  # True yaparsan Dueling DQN olur
)
qnet = train_dqn(dqn_train_cfg, DQNConfig(double_dqn=True))
print('\nDQN eğitim bitti. Checkpoint:', 'nb_ckpt_dqn/latest.pt')

### Dueling varyantı

Aynı eğitim, sadece `dueling=True`. State value `V(s)` global tokendan okunur, her dal advantage olarak öğrenilir.

In [ ]:
duel_cfg = DQNTrainConfig(
    iterations=3,
    episodes_per_iter=1,
    max_episode_steps=80,
    d_model=32, n_heads=2, n_layers=2,
    batch_size=32, learn_starts=32, replay_capacity=2000,
    target_sync_interval=50, eps_decay_steps=500,
    checkpoint_every=3, checkpoint_dir='nb_ckpt_duel',
    log_path='nb_ckpt_duel/train_log.jsonl',
    seed=0,
    dueling=True,
)
duel_qnet = train_dqn(duel_cfg, DQNConfig(double_dqn=True))
print('\nDueling DQN bitti. Checkpoint:', 'nb_ckpt_duel/latest.pt')

## 4. Oyunu İzle

`play_self_play_game` checkpoint yüklenmiş bir net'i alır, kendine karşı oynatır, ve `render=True` ise her `render_every` turda matplotlib ile board çizer.

Render notebook hücresinde inline çıkar — terminalde matplotlib popup açılır.

In [ ]:
# PPO checkpoint'ini yükle ve oyna
net, kind, ckpt = load_any_checkpoint('nb_ckpt_ppo/latest.pt')
print(f'Yüklenen: {kind.upper()}, {sum(p.numel() for p in net.parameters()):,} param')

summary = play_self_play_game(
    net,
    max_steps=100,
    render=True,
    render_every=30,
    seed=42,
    deterministic=False,
)
summary['history'] = f'<{len(summary["history"])} steps suppressed>'
print(summary)

DQN checkpoint'ini de aynı API ile yükleyebilirsin — `load_any_checkpoint` PPO ve DQN'i otomatik ayırır.

In [ ]:
net, kind, ckpt = load_any_checkpoint('nb_ckpt_dqn/latest.pt')
print(f'Yüklenen: {kind.upper()} (dueling={ckpt["arch"].get("dueling", False)})')

summary = play_self_play_game(
    net, max_steps=100, render=True, render_every=30, seed=42,
)

## 5. Eğitim Loglarını Görselleştir

Her iter sonunda training loop bir JSON satırı yazar (`train_log.jsonl`). Buradan kayıp eğrisi çıkarırız.

In [ ]:
import json
from pathlib import Path

def load_log(path):
    p = Path(path)
    if not p.exists():
        return []
    return [json.loads(line) for line in p.read_text().splitlines() if line.strip()]

ppo_logs = load_log('nb_ckpt_ppo/train_log.jsonl')
dqn_logs = load_log('nb_ckpt_dqn/train_log.jsonl')
duel_logs = load_log('nb_ckpt_duel/train_log.jsonl')

fig, axs = plt.subplots(1, 3, figsize=(15, 4))
if ppo_logs:
    iters = [l['iter'] for l in ppo_logs]
    axs[0].plot(iters, [l['policy_loss'] for l in ppo_logs], label='policy', marker='o')
    axs[0].plot(iters, [l['value_loss'] for l in ppo_logs], label='value', marker='s')
    axs[0].set_title('PPO losses'); axs[0].legend(); axs[0].set_xlabel('iter')
if dqn_logs:
    iters = [l['iter'] for l in dqn_logs]
    axs[1].plot(iters, [l['dqn_loss'] for l in dqn_logs], label='loss', marker='o')
    axs[1].plot(iters, [l['td_error'] for l in dqn_logs], label='td_err', marker='s')
    axs[1].set_title('DQN losses'); axs[1].legend(); axs[1].set_xlabel('iter')
if duel_logs:
    iters = [l['iter'] for l in duel_logs]
    axs[2].plot(iters, [l['dqn_loss'] for l in duel_logs], label='loss', marker='o')
    axs[2].plot(iters, [l['q_mean'] for l in duel_logs], label='q_mean', marker='s')
    axs[2].set_title('Dueling DQN'); axs[2].legend(); axs[2].set_xlabel('iter')
plt.tight_layout(); plt.show()

## 6. Üretim Eğitimi İçin Tavsiyeler

Notebook tiny config eğitim göstermek içindir; gerçek eğitim için aşağıdakileri CLI'dan başlat:

**PPO baseline:**
```bash
python -m rl.train \
    --iterations 500 \
    --episodes-per-iter 8 \
    --max-episode-steps 500 \
    --d-model 128 --n-heads 4 --n-layers 4 \
    --batch-size 256 --epochs 4 \
    --entropy-coef 0.01 \
    --planet-diff-weight 0.01 --ship-diff-weight 0.0005
```

**DQN baseline:**
```bash
python -m rl.dqn_train \
    --iterations 500 \
    --episodes-per-iter 4 \
    --d-model 128 --n-layers 4 \
    --replay-capacity 200000 \
    --learn-starts 5000 \
    --batch-size 256 \
    --eps-decay-steps 100000
```

**Dueling DQN**: yukarıdakine `--dueling` ekle.

### İzleme ipuçları
- `train_log.jsonl` her iter'i JSON olarak yazar; `pandas.read_json(..., lines=True)` ile okuyabilirsin.
- `tensorboard` istersen `train.py`'a `SummaryWriter` ekleyebiliriz (şu an sade JSON).
- Win-rate dengelendiyse (P0 ≈ P1) self-play sağlıklı; biri çok baskınsa learning collapse olabilir → frozen opponent pool eklemeyi düşün.
- `clip_fraction > 0.3` PPO'da güncellemenin çok agresif olduğu sinyali; lr veya clip_range küçült.
- DQN'de `q_mean` patlarsa (>10) gamma çok yüksek veya reward shape çok büyük; kontrol et.